**Tabela** | ecommerce_categorias |

**Origem** | squad2/silver/ecommerce_categorias (Delta) |

**Destinos** | 
* `gold/ecommerce_categorias_kpi` (Lakehouse - Overwrite)
* `gold/ecommerce_categorias_historico` (Lakehouse - Append)
* SQL Server: `ecommerce_categorias_kpi` (Overwrite) e `ecommerce_categorias_historico` (Append)
**Modo** | Delta Incremental Otimizado (Via Native SQL Server Connector)
**Regra de Negócio Aplicada** |
* Regra 2: Alertar se o número total de categorias raiz mudar entre lotes (adição/remoção não planejada)

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_categorias"
TABELA_SQL = f"gold_{TABELA}"  

path_silver  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}"
path_control = f"gold/control/{TABELA}.json"

try:
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de {TABELA} ainda não foi inicializada.")
    else:
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas()
        
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
        # df_novos_dados = df_pandas.copy() -- testes manuais
        
        if df_novos_dados.empty:
            print(" Camada Gold de Categorias em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} linhas para a Tabela Final...")
            
            # Regra de Negócio: Identificação de Categorias Raiz
            col_pai = [c for c in df_pandas.columns if 'pai' in c.lower() or 'parent' in c.lower()]
            if col_pai:
                col_pai = col_pai[0]
                df_novos_dados['is_categoria_raiz'] = df_novos_dados.apply(
                    lambda row: 'SIM' if (pd.isna(row[col_pai]) or row[col_pai] == "") else 'NAO', axis=1
                )
            else:
                df_novos_dados['is_categoria_raiz'] = "N/A"
            
            #  Coluna de Auditoria e Verificação de Atualização
            df_novos_dados['gold_processed_at'] = datetime.now()
            
            for col in df_novos_dados.columns:
                if pd.api.types.is_datetime64_any_dtype(df_novos_dados[col]):
                    df_novos_dados[col] = df_novos_dados[col].dt.tz_localize(None)
            
            # SINK 1: Gravação Lakehouse (Pasta Gold)
            write_deltalake(path_gold, df_novos_dados, mode="append", storage_options=get_storage_options())
            
            # -------------------------------------------------------------------------
            # SINK 2: INGESTÃO NO SQL SERVER COM CRIAÇÃO E ALINHAMENTO AUTOMÁTICO
            # -------------------------------------------------------------------------
            try:
                df_schema_sql = spark.read \
                    .format("sqlserver") \
                    .options(**SQL_OPTIONS) \
                    .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                    .load() \
                    .limit(0)
                
                spark_df_final = spark.createDataFrame(df_novos_dados)
                for col_db in df_schema_sql.columns:
                    col_match = [c for c in spark_df_final.columns if c.lower() == col_db.lower()]
                    if col_match:
                        spark_df_final = spark_df_final.withColumnRenamed(col_match[0], col_db)
                spark_df_aligned = spark_df_final.select(*df_schema_sql.columns)
                print(f"  Tabela existente localizada. Alinhando colunas e fazendo Append...")
            except Exception:
                print(f"  Criando nova tabela diferenciada: [squad2].[{TABELA_SQL}]...")
                spark_df_aligned = spark.createDataFrame(df_novos_dados)
            
            spark_df_aligned.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                .mode("append") \
                .save()
            
            # Atualiza controle JSON
            arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print(f" SUCESSO! Dados de categorias gravados em squad2.{TABELA_SQL}!")

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise